In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter, iirnotch, welch
from datetime import datetime

def design_filters(fs):
    """Creates a bandpass and a notch filter based on sampling rate."""
    nyq = 0.5 * fs
    # Bandpass: 1Hz to 100Hz (Covers SSVEP range and 90Hz target)
    low, high = 1.0 / nyq, 100.0 / nyq
    b_band, a_band = butter(4, [low, high], btype='band')
    
    # Notch: Removes 60Hz power line noise
    b_notch, a_notch = iirnotch(60.0, 30, fs)
    return b_band, a_band, b_notch, a_notch

def calculate_snr(pxx, freqs, target):
    """Calculates Signal-to-Noise Ratio at a target frequency."""
    # Find power at the exact target frequency
    idx_target = np.argmin(np.abs(freqs - target))
    power_target = pxx[idx_target]
    
    # Calculate average noise power in a +/- 4Hz window (excluding target)
    mask_noise = (freqs >= (target - 4)) & (freqs <= (target + 4))
    mask_target = (freqs >= (target - 0.5)) & (freqs <= (target + 0.5))
    noise_power = np.mean(pxx[mask_noise & ~mask_target])
    
    # Return SNR in decibels
    return 10 * np.log10(power_target / noise_power)

def process_session(eeg_path, log_json_path, sync_csv_path):
    # 1. Load All Data
    eeg_df = pd.read_csv(eeg_path)
    stim_sync_df = pd.read_csv(sync_csv_path)
    with open(log_json_path, 'r') as f:
        log_data = json.load(f)

    # 2. Synchronize Clocks (Browser performance.now vs System Unix)
    # Using the median offset from the sync file
    offsets = stim_sync_df['write_ts_ms'] - stim_sync_df['event_type']
    time_offset = np.median(offsets)
    
    # 3. Setup Parameters
    # Get sampling rate from BCI device timestamps
    fs = 1000.0 / np.mean(np.diff(eeg_df['device_ts_ms']))
    b_b, a_b, b_n, a_n = design_filters(fs)
    
    # Get targets from the caretaker's JSON config
    targets = [opt['freq'] for opt in log_data['config']['options']]
    # Filter targets that are above Nyquist (fs/2)
    valid_targets = [t for t in targets if t < (fs / 2)]
    
    print(f"Sampling Rate: {fs:.2f} Hz | Targets Found: {valid_targets} Hz")

    # 4. Analyze All Channels
    channels = [f'EEG_{i}' for i in range(1, 9)]
    results = []

    plt.figure(figsize=(15, 10))
    
    for i, ch in enumerate(channels):
        # Filter the raw signal
        raw = eeg_df[ch].values - np.mean(eeg_df[ch].values)
        clean = lfilter(b_b, a_b, raw)
        clean = lfilter(b_n, a_n, clean)
        
        # Frequency Analysis (PSD)
        f, pxx = welch(clean, fs, nperseg=int(fs*4))
        
        # Check SNR for the first target (e.g. 90Hz)
        primary_target = valid_targets[0]
        snr = calculate_snr(pxx, f, primary_target)
        results.append({'Channel': ch, 'SNR_dB': snr})
        
        # Plotting
        plt.subplot(4, 2, i+1)
        plt.semilogy(f, pxx)
        plt.axvline(primary_target, color='red', linestyle='--', alpha=0.6)
        plt.title(f"{ch} (SNR: {snr:.2f} dB)")
        plt.xlim(0, 110)
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.suptitle(f"Multi-Channel Analysis for {primary_target}Hz Stimulus", y=1.02, fontsize=16)
    plt.savefig('session_analysis.png')

    # 5. Export Ranked Results
    rankings_df = pd.DataFrame(results).sort_values(by='SNR_dB', ascending=False)
    rankings_df.to_csv('channel_rankings.csv', index=False)
    
    best_ch = rankings_df.iloc[0]['Channel']
    print(f"Done! Best signal found on {best_ch}. Analysis saved to 'session_analysis.png'.")

# Run the pipeline
if __name__ == "__main__":
    process_session(
        eeg_path='',
        log_json_path='',
        sync_csv_path=''
    )